# The OptiPFair Series #2: Healing the Golden Scar

In our [second article](https://principia-agentica.com/blog/2026/03/22/healing-the-golden-scar-optipfair/) of the OptiPFair Series, we explored two massive architectural evolutions in Pere Martra's model optimization library:

1. **Hardware-Aware Alignment (`expansion_divisor`)**: Ensuring pruned tensors remain perfectly divisible by hardware-friendly numbers (like 64) for Tensor Core efficiency.
2. **Data-Driven Pruning (`dataloader`)**: Shifting from static weight analysis to dynamic, context-aware pruning using the Peak-to-Peak Magnitude (PPM) method.

This notebook is the practical proof of concept. We will take `meta-llama/Llama-3.2-1B`, expose its "golden scar" through naive static pruning, and then heal it by forging a highly specialized coding engine using `CodeAlpaca`.

## 1. The Ignition (Setup)
First, we install the necessary tools and load the base model.

In [ ]:
!pip install -q optipfair transformers datasets torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import optipfair as opf

MODEL_ID = "meta-llama/Llama-3.2-1B"

print(f"Loading baseline model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Ensure pad token is set for dataloader batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the model on CPU for the PoC (or CUDA if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16).to(device)

# Let's inspect the original intermediate size of the MLP layer
original_intermediate_size = model.model.layers[0].mlp.gate_proj.out_features
print(f"\nOriginal Llama-3.2-1B Intermediate Size: {original_intermediate_size}")
print(f"Divisible by 64? {original_intermediate_size % 64 == 0}")

## 2. The Baseline (The Jagged Edge)
Let's simulate the "golden scar." We will perform a naive, static width prune (cutting 20% of the neurons) without the `expansion_divisor`. 

Notice how the resulting tensor size fractures the hardware alignment.

In [ ]:
print("Forging baseline... Static pruning (20%) without hardware alignment.")

# Clone model to avoid modifying the base instance for later steps
import copy
static_model = copy.deepcopy(model)

static_pruned, static_stats = opf.prune_model(
    model=static_model,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW", # PPM method (static fallback without dataloader)
    pruning_percentage=20,
    show_progress=False,
    return_stats=True
)

static_intermediate_size = static_pruned.model.layers[0].mlp.gate_proj.out_features
print(f"\nJagged Intermediate Size: {static_intermediate_size}")
print(f"Divisible by 64? {static_intermediate_size % 64 == 0}")
print("Result: Tensor Cores will waste cycles padding this dimension.")



## 3. The Destiny (Calibration Data)
To forge a true specialist, we must give the model its destiny. We will load a subset of the `sahil2801/CodeAlpaca-20k` dataset. 

By passing this through the dataloader, the PPM method will measure which neurons actually light up when processing Python code. Neurons that only dream of French poetry will be starved and severed.

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader

print("Loading the CodeAlpaca destiny...")
dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train[:100]") # Tiny slice for PoC

def tokenize_function(examples):
    # Combine instruction and output for calibration context
    texts = [f"{i} {o}" for i, o in zip(examples["instruction"], examples["output"])]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")

print("Tokenizing calibration data...")
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
tokenized_dataset.set_format("torch")

# Create the DataLoader (The Silk)
dataloader = DataLoader(tokenized_dataset, batch_size=4)

## 4. The Engine (Healing the Scar)
Now, we strike the anvil perfectly. We run the hybrid, data-driven prune using our `dataloader` AND we apply the `expansion_divisor=64` to mathematically snap the resulting layers back into hardware alignment.

In [ ]:
print("Igniting the forge. Calibrating hybrid resonance and snapping to grid...")

specialized_model, specialized_stats = opf.prune_model(
    model=model,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW", # Invokes Hybrid PPM with dataloader
    pruning_percentage=20,
    expansion_divisor=64,          # HARDWARE ALIGNMENT
    dataloader=dataloader,         # DATA-DRIVEN CALIBRATION
    show_progress=True,
    return_stats=True
)

specialized_intermediate_size = specialized_model.model.layers[0].mlp.gate_proj.out_features
print(f"\nHealed Intermediate Size: {specialized_intermediate_size}")
print(f"Divisible by 64? {specialized_intermediate_size % 64 == 0}")
print("Result: The scar is healed. The machine sings in perfect rhythm.")

## 5. The Road Test (Benchmarking & Inference)
To truly understand the value of the forge, we must measure the results. We will compare:
1. **The Original Model**
2. **The Statically Pruned Model** (The baseline)
3. **The Data-Driven Specialized Model** (The healed scar)

We will measure parameter size reduction, tokens-per-second (TPS) performance, and evaluate the quality of a coding prompt.

In [ ]:
import time

def benchmark_model(name, eval_model, tokenizer, prompt, device):
    print(f"\n{'='*40}")
    print(f" 🏎️ Benchmarking: {name}")
    print(f"{'='*40}")
    
    # 1. Measure Parameters (Size Reduction)
    params = sum(p.numel() for p in eval_model.parameters())
    print(f"Parameters: {params:,}")
    
    # Prepare inputs
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Warmup pass (important for accurate GPU timing)
    with torch.no_grad():
        _ = eval_model.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)
        
    # 2. Timed generation (Performance Improvement)
    start_time = time.time()
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs, 
            max_new_tokens=60, 
            temperature=0.2, # Lower temperature for more deterministic coding output
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    end_time = time.time()
    
    generated_tokens = outputs[0].shape[0] - inputs.input_ids.shape[1]
    generation_time = end_time - start_time
    tps = generated_tokens / generation_time
    
    print(f"Speed: {tps:.2f} tokens/sec ({generation_time:.2f}s total)")
    print(f"\n--- Output ---")
    
    # 3. Output Quality
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    
    return params, tps

# A standard Python coding prompt
test_prompt = "def calculate_factorial(n):\n    \"\"\"Calculates the factorial of a number.\"\"\""

# Run Benchmarks
bench_original = benchmark_model("Original Llama-3.2-1B", model, tokenizer, test_prompt, device)
bench_static = benchmark_model("Static Pruned (20%)", static_pruned, tokenizer, test_prompt, device)
bench_specialist = benchmark_model("Data-Driven Specialist (20% + HW Aligned)", specialized_model, tokenizer, test_prompt, device)

# Final Analytical Summary
print(f"\n{'*'*40}")
print(f" 🏆 FINAL SUMMARY")
print(f"{'*'*40}")

size_reduction = 100 - (bench_specialist[0] / bench_original[0] * 100)
speedup = bench_specialist[1] / bench_original[1]

print(f"Size Reduction (Original -> Specialist): {size_reduction:.1f}% smaller")
print(f"Speedup (Original -> Specialist): {speedup:.2f}x faster inference")
